# JobMatch AI — Modelo de Regressão Salarial

Este notebook:
1. Carrega vagas processadas com salário
2. Vetoriza descrições com TF-IDF
3. Avalia GradientBoosting e RandomForest via RMSE
4. Treina o melhor modelo
5. Testa predição em exemplo

---

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

from src.pipeline.preprocess import clean_text
from src.models.vectorizer import fit_vectorizer, transform
from src.models.salary_model import train_salary_model, predict_salary_range

print(f"📂 Projeto: {project_root}")

---
## 1. Carregar Dados Processados

In [ ]:
PROC = project_root / 'data' / 'processed'
jobs_path = PROC / 'jobs_clean.parquet'

if not jobs_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {jobs_path}")

jobs = pd.read_parquet(jobs_path)
print(f"✅ Jobs carregados: {jobs.shape}")

---
## 2. Filtrar Vagas com Salário

In [ ]:
jobs_with_sal = jobs.dropna(subset=['salary_annual_avg']).copy()
jobs_with_sal = jobs_with_sal[jobs_with_sal['salary_annual_avg'] > 0]
print(f"📊 Vagas com salário válido: {len(jobs_with_sal):,}")
print(f"   Salário médio: ${jobs_with_sal['salary_annual_avg'].mean():,.0f}")
print(f"   Mediana: ${jobs_with_sal['salary_annual_avg'].median():,.0f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
jobs_with_sal['salary_annual_avg'].hist(bins=50)
plt.title('Distribuição do Target')
plt.xlabel('Salário Anual (USD)')

plt.subplot(1, 2, 2)
plt.boxplot(jobs_with_sal['salary_annual_avg'])
plt.title('Boxplot do Target')
plt.tight_layout()
plt.show()

---
## 3. Vetorização e Treino

In [ ]:
corpus = jobs_with_sal['full_text'].dropna().tolist()
print(f"📚 Corpus: {len(corpus)} documentos")

vec = fit_vectorizer(corpus)
X = transform(corpus, vec)
y = jobs_with_sal.loc[jobs_with_sal['full_text'].notna(), 'salary_annual_avg'].values
print(f"✅ Matriz TF-IDF: {X.shape}")

In [ ]:
model = train_salary_model(X, y)

---
## 4. Teste com Exemplo

In [ ]:
exemplo = jobs_with_sal.iloc[0]
job_vec = transform([exemplo['full_text']], vec)
pred = predict_salary_range(job_vec, model)

print(f"🎯 Exemplo: {exemplo['title']}")
print(f"   Salário real:    ${exemplo['salary_annual_avg']:,.0f}")
print(f"   Salário predito: ${pred['estimated_annual_usd']:,.0f}")
print(f"   Faixa estimada:  ${pred['range_low']:,.0f} – ${pred['range_high']:,.0f}")
erro = abs(pred['estimated_annual_usd'] - exemplo['salary_annual_avg'])
print(f"   Erro absoluto:   ${erro:,.0f} ({erro/exemplo['salary_annual_avg']*100:.1f}%)")

In [ ]:
# Predição em lote no conjunto de teste
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model_test = train_salary_model(X_tr, y_tr)
y_pred = model_test.predict(X_te)

plt.figure(figsize=(8, 6))
plt.scatter(y_te, y_pred, alpha=0.3)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--')
plt.xlabel('Salário Real (USD)')
plt.ylabel('Salário Predito (USD)')
plt.title('Predito vs Real - Conjunto de Teste')
plt.tight_layout()
plt.show()

from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
print(f"📊 RMSE no teste: ${rmse:,.0f}")